# graphArt_LM: composing node lists and CoS wiring (Qwen 14B, torchvision 21)

Three jobs, inference only, on Ollama:
1. **E1**: the 14B wires the graph with the CoS prompt, handed the node lists that the
   fine-tuned 1.5B (mixed-v2) wrote, instead of the true ones.
2. **E3**: the same prompt on the true nodes, listed in a random valid order so parallel
   branches interleave (two seeds; only the 11 samples whose order can change).

About 1.75 hours, capped at 2.5. Scoring happens on the laptop.

**Before running**: *GPU T4 x2*, *Internet on*. No inputs. Then
*Save Version -> Save & Run All (Commit)*. When it finishes, download
`graphart_results.zip` from the *Output* tab.

Generated by `scripts/make_kaggle_notebook.py --probe-14b-compose`. Edit that script, not this file.

In [ ]:
# JOBS         run in order, highest value first, so a session that runs out of time loses
#              the least. Each is one model x prompt x data set: split is test, torchvision
#              or edits; stratify 10 takes the same 40 test programs as every Phase 1 run
#              (None = the whole set); max_tokens None keeps the runner's 1500-token cap.
# MODELS       pulled in order. qwen2.5-coder:14b fits one T4; gemma4:31b needs both.
# TIME_BUDGET  stop starting new work with an hour of the 12 to spare.
# SMOKE_ONLY   True: install, download, try one sample, and stop.
# VALIDATE_ONLY True: only check the code arrived intact. Run it on a CPU session
#              (Accelerator: None), which costs no GPU quota.
JOBS = [{'model': 'qwen2.5-coder:14b', 'prompt': 'edges_only_cos', 'split': 'torchvision', 'stratify': None, 'max_tokens': 6000, 'num_ctx': 16384, 'extra': ['--nodes-from', 'data/step1_given/hf__qwen2.5-coder-1.5b-graphart-lora-mixed-v2__zero_shot__torchvision.jsonl']}, {'model': 'qwen2.5-coder:14b', 'prompt': 'edges_only_cos', 'split': 'torchvision', 'stratify': None, 'max_tokens': 6000, 'num_ctx': 16384, 'extra': ['--nodes-from', 'data/step1_given/reference__shuffled1__torchvision.jsonl']}, {'model': 'qwen2.5-coder:14b', 'prompt': 'edges_only_cos', 'split': 'torchvision', 'stratify': None, 'max_tokens': 6000, 'num_ctx': 16384, 'extra': ['--nodes-from', 'data/step1_given/reference__shuffled2__torchvision.jsonl']}]; MODELS = list(dict.fromkeys(j['model'] for j in JOBS)); TIME_BUDGET_HOURS = 2.5; SMOKE_ONLY = False; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "5f14abe620af0da0b44f2c8d34a4636202c8af347c1ff7c082b50eba0c59304b"
BUNDLE = (
    "..."
    "wFc0dVXkscRn+wIAAAAABFla"
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install and start Ollama

In [ ]:
def ollama_up() -> bool:
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read().decode())
        return True
    except OSError:
        return False

if not shutil.which("ollama"):
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null",
                   shell=True, check=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

if not ollama_up():
    # Ollama splits a model over both T4s by itself when it does not fit on one.
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="24h")
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open(WORK / "ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_up():
            break
    else:
        raise RuntimeError(f"ollama did not start -- see {WORK / 'ollama.log'}")

for model in MODELS:
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"])

## 3. Smoke test: one sample, before committing hours to it

In [ ]:
# Shows the reply, how long it took and how long it is, so a model that thinks out loud
# or ignores the format is obvious before the full run.
from pipeline.backends import make_backend
from pipeline.run_prompting import load_prompt, load_split, _render

row = load_split("test", None, None, 10)[0]
system, template = load_prompt("zero_shot")
for model in MODELS:
    t = time.time()
    reply = make_backend("ollama", model)(system, _render(template, row))
    took = time.time() - t
    print(f"=== {model}: {took:.0f}s, {len(reply)} chars\n{reply[:1500]}")
    # GPU hours are scarce: a model that does not answer in the format would burn the
    # whole session producing nothing scorable, so stop here instead.
    if "NODES" not in reply or "EDGES" not in reply:
        raise SystemExit(f"{model}: the smoke-test reply is not a graph -- stopping before the full run")
    # Longer answers (cot, cos, torchvision) take several times this; a lower bound.
    print(f"{model}: {took:.0f} s for one zero_shot test answer")

## 4. The run

In [ ]:
def remaining() -> float:
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package() -> None:
    # Re-zipped after every prompt, so a session that dies late keeps what it finished.
    if (REPO / "results").exists():
        shutil.make_archive(str(WORK / "graphart_results"), "zip", REPO, "results")

for j in ([] if SMOKE_ONLY else JOBS):
    name = f"{j['model']} / {j['prompt']} / {j['split']}" + (f" ({j['stratify']} per tier)" if j["stratify"] else "")
    if remaining() < 600:
        print(f"time budget spent -- skipping {name}")
        continue
    cmd = [sys.executable, "-m", "pipeline.run_prompting", "--backend", "ollama",
           "--model", j["model"], "--prompt", j["prompt"], "--split", j["split"]]
    if j["stratify"]:
        cmd += ["--stratify", str(j["stratify"])]
    if j["max_tokens"]:
        cmd += ["--max-tokens", str(j["max_tokens"])]
    if j.get("num_ctx"):
        cmd += ["--num-ctx", str(j["num_ctx"])]
    if j.get("limit"):                      # for trying a notebook on a few samples
        cmd += ["--limit", str(j["limit"])]
    cmd += j.get("extra", [])                # e.g. --nodes-from, --shuffle-nodes
    name += f" {' '.join(j.get('extra', []))}"
    print(f"\n##### {name}   ({remaining() / 3600:.1f} h left)", flush=True)
    try:
        # Cached replies survive a timeout; a later session would only redo the rest.
        subprocess.run(cmd, check=False, timeout=remaining(), env=dict(os.environ, PYTHONUNBUFFERED="1"))
    except subprocess.TimeoutExpired:
        print("stopped by the time budget")
    package()

package()
print(f"\ndone in {(time.time() - START) / 3600:.1f} h")
raw_dir = REPO / "results" / "raw"
for run in sorted(raw_dir.iterdir()) if raw_dir.exists() else []:
    print(f"{run.name:55s} {len(list(run.glob('*.txt'))):3d} replies")
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:55s} n={m['n']} failed={m['failed']}")
if not SMOKE_ONLY:
    print(f"download {WORK / 'graphart_results.zip'} from the Output tab")